# 02 · Baseline (M0), overfit smoke test and LoRA pilot

This notebook runs the first experiment end to end on one Kaggle T4, using the package CLIs:

1. **Data**: prepare ScienceQA image questions (`visiontune.data.prepare`).
2. **M0 baseline**: the original Qwen2.5-VL-3B on a fixed 500-question validation subset.
3. **Smoke test**: LoRA on 8 examples for 15 epochs. The loss must collapse (blueprint §5.6). The notebook **stops here**
   if it doesn't.
4. **LoRA pilot (D1)**: 500 training examples, 2 epochs, A1 targets (`configs/lora.yaml`).
5. **Pilot evaluation**: the same 500 validation questions, with a paired comparison against M0.

This is validation data only. The ScienceQA test split is not touched (blueprint §5.6).
Settings: Accelerator → GPU T4, Internet → On. Outputs go to `/kaggle/working/run02/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/akhilnathreddy/VisionTune.git"
BRANCH = "main"
REPO_DIR = "/tmp/VisionTune"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

In [ ]:
!pip install -q -U "transformers>=4.57" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.45" qwen-vl-utils
!pip install -q -e .

In [ ]:
import json, time
from pathlib import Path

OUT = Path("/kaggle/working/run02")
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "outputs").mkdir(parents=True, exist_ok=True)
N_VAL = 500
timings = {}

def sh(cmd):
    """Run a CLI step, stream its output, fail the notebook on error, record wall time."""
    print("$", cmd, flush=True)
    t0 = time.perf_counter()
    subprocess.run(cmd, shell=True, check=True)
    return round(time.perf_counter() - t0, 1)

## 1. Data

In [ ]:
timings["prepare_s"] = sh("python -m visiontune.data.prepare --config configs/base.yaml --skip-ai2d > /dev/null")
report = json.loads(Path("data/processed/prepare_report.json").read_text())
print({s: v["kept"] for s, v in report["scienceqa"]["splits"].items()}, timings)

## 2. M0 baseline on the validation subset

In [ ]:
base_pred = OUT / "predictions" / f"base_scienceqa_validation_n{N_VAL}.jsonl"
timings["baseline_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/base.yaml "
                                f"--split validation --limit {N_VAL} --out {base_pred}")
base_metrics = json.loads(base_pred.with_suffix(".metrics.json").read_text())
print(json.dumps({k: base_metrics[k] for k in ("overall", "format", "latency_s_per_example_batched", "peak_vram_gb")}, indent=1))

## 3. Overfit smoke test (8 examples)

In [ ]:
timings["smoke_train_s"] = sh(f"python -m visiontune.training.train --config configs/smoke.yaml train.output_root={OUT}/outputs")
smoke = json.loads((OUT / "outputs/smoke_overfit_lora/run_summary.json").read_text())
losses = [h["loss"] for h in smoke["log_history"] if "loss" in h]
print("loss trajectory:", [round(x, 3) for x in losses])
print("peak VRAM:", smoke["peak_vram_gb"], "GB | trainable:", smoke["parameters"]["trainable"])
assert losses[-1] < 0.3 * losses[0], "Smoke test failed: loss did not collapse on 8 examples - stop and debug"

## 4. LoRA pilot: 500 examples, A1 targets

In [ ]:
timings["pilot_train_s"] = sh(f"python -m visiontune.training.train --config configs/lora.yaml train.output_root={OUT}/outputs")
run_dir = OUT / "outputs/scienceqa_qwen25vl3b_lora_r16_seed42"
pilot = json.loads((run_dir / "run_summary.json").read_text())
print(json.dumps({k: pilot[k] for k in ("train_examples", "train_seconds_this_session", "peak_vram_gb",
                                        "adapter_size_mb", "lora_targets", "parameters", "final_eval")}, indent=1))
print("train loss:", [round(h["loss"], 3) for h in pilot["log_history"] if "loss" in h])
print("eval loss:", [round(h["eval_loss"], 3) for h in pilot["log_history"] if "eval_loss" in h])

## 5. Pilot evaluation and paired comparison with M0

In [ ]:
lora_pred = OUT / "predictions" / f"lora_pilot_scienceqa_validation_n{N_VAL}.jsonl"
timings["pilot_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/lora.yaml "
                             f"--adapter {run_dir}/adapter --split validation --limit {N_VAL} --out {lora_pred}")
lora_metrics = json.loads(lora_pred.with_suffix(".metrics.json").read_text())

from visiontune.data.io import read_jsonl
from visiontune.evaluation.metrics import mcnemar_exact, paired_bootstrap

base = {p["id"]: p["correct"] for p in read_jsonl(base_pred)}
lora = {p["id"]: p["correct"] for p in read_jsonl(lora_pred)}
ids = sorted(base)
assert ids == sorted(lora), "baseline and pilot must be scored on identical examples"
a, b = [base[i] for i in ids], [lora[i] for i in ids]
comparison = {
    "n": len(ids),
    "base_accuracy": base_metrics["overall"], "lora_accuracy": lora_metrics["overall"],
    "base_format": base_metrics["format"], "lora_format": lora_metrics["format"],
    "paired_bootstrap_lora_minus_base": paired_bootstrap(a, b),
    "mcnemar": mcnemar_exact(a, b),
    "by_subject": {"base": base_metrics["by_subject"], "lora": lora_metrics["by_subject"]},
    "by_near_dup_in_train": {"base": base_metrics["by_near_dup_in_train"], "lora": lora_metrics["by_near_dup_in_train"]},
    "latency_s_per_example_batched": {"base": base_metrics.get("latency_s_per_example_batched"),
                                      "lora": lora_metrics.get("latency_s_per_example_batched")},
    "timings_s": timings, "commit": COMMIT,
}
(OUT / "comparison_base_vs_lora_pilot.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps({k: comparison[k] for k in ("n", "base_accuracy", "lora_accuracy", "base_format", "lora_format",
                                             "paired_bootstrap_lora_minus_base", "mcnemar", "timings_s")}, indent=1))

In [ ]:
# Keep the adapters and summaries; drop resumable checkpoints (optimizer state) to keep the output small.
import shutil
for ckpt in (OUT / "outputs").glob("*/checkpoint-*"):
    shutil.rmtree(ckpt)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")